# Seminar 6 - Language Modelling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm.auto import tqdm, trange
import re

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [ ]:
trump_df = pd.read_csv('/kaggle/input/trump-tweets-2009-2025/djt_posts_dec2025.csv')
trump_df


,id,date,platform,handle,text,favorite_count,repost_count,quote_flag,repost_flag,deleted_flag,word_count,hashtags,urls,user_mentions,media_count,media_urls,post_url,in_reply_to
0,115816402893182666,2025-12-31 21:54:21+00:00,Truth Social,realDonaldTrump,"Good News! George and Amal Clooney, two of the...",52,16,False,False,False,144,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
1,115816171466225987,2025-12-31 20:55:30+00:00,Truth Social,realDonaldTrump,We are removing the National Guard from Chicag...,44,16,False,False,False,107,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
2,115816098525451632,2025-12-31 20:36:57+00:00,Truth Social,realDonaldTrump,The Democrats are a bunch of cheaters and thie...,461,179,False,False,False,50,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
3,115816063544553030,2025-12-31 20:28:03+00:00,Truth Social,realDonaldTrump,Republicans: No more money to Fat Cat Insuranc...,309,97,False,False,False,24,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
4,115815971019162695,2025-12-31 20:04:31+00:00,Truth Social,realDonaldTrump,The United States has set a World Record on in...,36,11,False,False,False,76,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90338,1773561338,2009-05-12 14:07:28+00:00,Twitter,realDonaldTrump,"""My persona will never be that of a wallflower...",1904,1345,False,False,False,21,NaN,NaN,NaN,0,NaN,https://x.com/realdonaldtrump/status/1773561338,NaN
90339,1741160716,2009-05-08 20:40:15+00:00,Twitter,realDonaldTrump,New Blog Post: Celebrity Apprentice Finale and...,24,11,False,False,False,13,NaN,http://www.trumpuniversity.com/blog/post/2009/...,NaN,0,NaN,https://x.com/realdonaldtrump/status/1741160716,NaN
90340,1737479987,2009-05-08 13:38:08+00:00,Twitter,realDonaldTrump,Donald Trump reads Top Ten Financial Tips on L...,33,15,False,False,False,17,NaN,https://www.youtube.com/watch?v=hmMkZD4VcNQ&fe...,NaN,0,NaN,https://x.com/realdonaldtrump/status/1737479987,NaN
90341,1701461182,2009-05-05 01:00:10+00:00,Twitter,realDonaldTrump,Donald Trump will be appearing on The View tom...,259,34,False,False,False,22,NaN,NaN,NaN,0,NaN,https://x.com/realdonaldtrump/status/1701461182,NaN


In [ ]:
trump_df['text'].sample(10, random_state=42).values


array(['https:// jonathanturley.org/2024/03/21/ the-perversity-of-michael-cohen-federal-judge-denounces-cohen-as-a-serial-perjurer/',
       'Just received a full briefing on the tragic shooting at NAS Pensacola in Florida, and spoke to @GovRonDeSantis. My thoughts and prayers are with the victims and their families during this difficult time. We are continuing to monitor the situation as the investigation is ongoing.',
       'Pelosi doesn’t want to hand over The Articles of Impeachment, which were fraudulently produced by corrupt politicians like Shifty Schiff in the first place, because after all of these years of investigations and persecution, they show no crimes and are a joke and a scam!',
       'Will be on Fox & Friends tomorrow morning at 7.00 - hope you enjoy!',
       '"@house6000: That limo story with that lucky guy changing the tire was amazing, I wish I was as lucky as he was, have a great day!" THANKS!',
       '"@AprilLaJune: OREGON votes today! Go vote for @realDonald

In [ ]:
trump_df['text'].isna().sum()


np.int64(3)

In [ ]:
trump_df = trump_df.dropna(subset=['text'])


In [ ]:
trump_df = trump_df.loc[(trump_df['text'] != '[Image]') & (trump_df['text'] != '[Video]') & (trump_df['text'] != '[QuickTime Video]')].reset_index()
trump_df.shape


(84829, 19)

In [ ]:
def clean(text):
    text = re.sub(r'^RT @\w+:?\s*', '', text) # remove retweets indicators
    text = re.sub(r'https?://\s*\S+', '', text) # remove links
    text = re.sub(r'www\.\s*\S+', '', text) # even broken ones
    text = re.sub(r'@\w+', '', text) # remove user tags
    text = text.replace('\xa0', ' ') # remove special whitespaces
    text = re.sub(r'\s+', ' ', text) # and redundant whitespaces
    text = re.sub(r"([^a-zA-Z0-9\s])", r" \1 ", text) # separate words and punctuation
    return text.lower().strip()


In [ ]:
trump_df['clean_text'] = trump_df['text'].apply(clean)
trump_df.head()


,index,id,date,platform,handle,text,favorite_count,repost_count,quote_flag,repost_flag,deleted_flag,word_count,hashtags,urls,user_mentions,media_count,media_urls,post_url,in_reply_to,clean_text
0,0,115816402893182666,2025-12-31 21:54:21+00:00,Truth Social,realDonaldTrump,"Good News! George and Amal Clooney, two of the...",52,16,False,False,False,144,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN,"good news ! george and amal clooney , two of..."
1,1,115816171466225987,2025-12-31 20:55:30+00:00,Truth Social,realDonaldTrump,We are removing the National Guard from Chicag...,44,16,False,False,False,107,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN,we are removing the national guard from chicag...
2,2,115816098525451632,2025-12-31 20:36:57+00:00,Truth Social,realDonaldTrump,The Democrats are a bunch of cheaters and thie...,461,179,False,False,False,50,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN,the democrats are a bunch of cheaters and thie...
3,3,115816063544553030,2025-12-31 20:28:03+00:00,Truth Social,realDonaldTrump,Republicans: No more money to Fat Cat Insuranc...,309,97,False,False,False,24,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN,republicans : no more money to fat cat insura...
4,4,115815971019162695,2025-12-31 20:04:31+00:00,Truth Social,realDonaldTrump,The United States has set a World Record on in...,36,11,False,False,False,76,NaN,NaN,NaN,0,NaN,https://truthsocial.com/@realDonaldTrump/posts...,NaN,the united states has set a world record on in...


In [ ]:
trump_df = trump_df[['text', 'clean_text']]
trump_df.head()


,text,clean_text
0,"Good News! George and Amal Clooney, two of the...","good news ! george and amal clooney , two of..."
1,We are removing the National Guard from Chicag...,we are removing the national guard from chicag...
2,The Democrats are a bunch of cheaters and thie...,the democrats are a bunch of cheaters and thie...
3,Republicans: No more money to Fat Cat Insuranc...,republicans : no more money to fat cat insura...
4,The United States has set a World Record on in...,the united states has set a world record on in...


In [ ]:
trump_df = trump_df[trump_df['clean_text'].str.len() > 5]
len(trump_df)


81901

In [ ]:
trump_df['word_count'] = trump_df['clean_text'].apply(lambda x: len(x.split()))
print(trump_df['word_count'].describe())


count    81901.000000
mean        32.073992
std         32.036907
min          1.000000
25%         16.000000
50%         24.000000
75%         34.000000
max        605.000000
Name: word_count, dtype: float64


In [ ]:
trump_df = trump_df[trump_df['word_count'] >= 15]
len(trump_df)


64537

#### Split data

In [ ]:
train_df, val_df = train_test_split(trump_df, test_size=0.1, random_state=42)


After we've done with data cleaning, it's time to create a vocabulary

In [ ]:
all_tokens = []
for text in train_df['clean_text']:
    all_tokens.extend(text.split())

word_counts = Counter(all_tokens)
len(word_counts)


38970

In [ ]:
vocab_size = 15000
common_words = [word for word, count in word_counts.most_common(vocab_size)]

word2idx = {word: i + 2 for i, word in enumerate(common_words)}
word2idx['<PAD>'] = 0
word2idx['<UNK>'] = 1
# We create a second vocab for convenience
idx2word = {i: word for word, i in word2idx.items()}
actual_vocab_size = len(word2idx)


In [ ]:
def create_sequences(df, word2idx, seq_length):
    input_seqs = []
    target_words = []
    pad_idx = word2idx['<PAD>']
    
    for text in df['clean_text']:
        words = text.split()
        indices = [word2idx.get(w, word2idx['<UNK>']) for w in words]

        if len(indices) <= seq_length:
            in_part = indices[:-1]
            target = indices[-1]

            padding = [pad_idx] * (seq_length - len(in_part))
            input_seqs.append(padding + in_part)
            target_words.append(target)

        else:
            for i in range(len(indices) - seq_length):
                input_seqs.append(indices[i : i + seq_length])
                target_words.append(indices[i + seq_length])
            
    return torch.tensor(input_seqs, dtype=torch.long), torch.tensor(target_words, dtype=torch.long)


In [ ]:
WINDOW_SIZE = 20

X_train, y_train = create_sequences(train_df, word2idx, WINDOW_SIZE)
X_val, y_val = create_sequences(val_df, word2idx, WINDOW_SIZE)


In [ ]:
class TrumpDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
train_loader = DataLoader(TrumpDataset(X_train, y_train), batch_size=256, shuffle=True)
val_loader = DataLoader(TrumpDataset(X_val, y_val), batch_size=256, shuffle=False)


#### Model

In [ ]:
class TrumpGenerator(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, n_layers=2):
        super(TrumpGenerator, self).__init__()
        
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim
        
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        # x: [batch_size, seq_len]
        x = self.embedding(x) # [batch_size, seq_len, emb_dim]
        # if hidden is None, it is defined as zeros
        out, hidden = self.lstm(x, hidden) 
        # We need to predict the next word
        # We take the last step: out[:, -1, :]
        last_time_step = out[:, -1, :]
        logits = self.fc(last_time_step)
        return logits, hidden


In [ ]:
EMB_DIM = 256
HIDDEN_DIM = 512
model = TrumpGenerator(actual_vocab_size, EMB_DIM, HIDDEN_DIM).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
from torchinfo import summary


In [ ]:
summary(model, input_size=(1, EMB_DIM), dtypes=[torch.long], device=device)


Layer (type:depth-idx)                   Output Shape              Param #
TrumpGenerator                           [1, 15002]                --
├─Embedding: 1-1                         [1, 256, 256]             3,840,512
├─LSTM: 1-2                              [1, 256, 512]             3,678,208
├─Linear: 1-3                            [1, 15002]                7,696,026
Total params: 15,214,746
Trainable params: 15,214,746
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 953.16
Input size (MB): 0.00
Forward/backward pass size (MB): 1.69
Params size (MB): 60.86
Estimated Total Size (MB): 62.55

In [ ]:
import torch.nn.functional as F


In [ ]:
def generate(model, seed_text, word2idx, idx2word, max_len=40, temperature=1.0):
    model.eval()
    
    # Indexate input text
    words = clean(seed_text).split()
    current_seq = [word2idx.get(w, word2idx['<UNK>']) for w in words]

    if len(current_seq) <= 20:
            padding = [word2idx['<PAD>']] * (20 - len(current_seq))
            current_seq = padding + current_seq
    
    hidden = None
    
    result = words
    
    with torch.no_grad():
        for _ in range(max_len):
            # prepare input tensor
            # Take last n words where n = WINDOW_SIZE
            input_tensor = torch.tensor([current_seq[-WINDOW_SIZE:]]).to(device)
            
            # predict
            logits, hidden = model(input_tensor, hidden)
            
            # Apply temperature
            logits = logits / max(temperature, 1e-6)
            
            probs = F.softmax(logits, dim=-1)
            
            # Sampling
            # We don't want to always predict the most probable word, but one of them to make generation diverse
            next_word_idx = torch.multinomial(probs, num_samples=1).item()
            
            # Add prediction to sequence
            next_word = idx2word[next_word_idx]
            result.append(next_word)
            current_seq.append(next_word_idx)
            
            # We can define custom stopping or postprocessing rules
            
    return " ".join(result)


In [ ]:
test_seeds = [
    "I will",
    "The fake news",
    "Make America"
]


In [ ]:
import copy


In [ ]:
epochs = 20
best_val_loss = float('inf')
patience = 3
counter = 0

history = {'train_loss': [], 'val_loss': []}

for epoch in trange(epochs):
    # --- TRAINING ---
    model.train()
    train_accum_loss = 0
    
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        logits, _ = model(x_batch)
        
        loss = criterion(logits, y_batch)
        loss.backward()
        
        # Add Gradient Clipping, so LSTM's gradients don't explode
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        
        optimizer.step()
        train_accum_loss += loss.item()
    
    avg_train_loss = train_accum_loss / len(train_loader)
    
    # --- VALIDATION ---
    model.eval()
    val_accum_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits, _ = model(x_batch)
            loss = criterion(logits, y_batch)
            val_accum_loss += loss.item()
            
    avg_val_loss = val_accum_loss / len(val_loader)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    
    print(f"\nEpoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # --- TEST GENERATION ---
    print("-" * 30)
    print("Generation samples:")
    for seed in test_seeds:
        generated = generate(model, seed, word2idx, idx2word, max_len=15, temperature=0.8)
        print(f"Seed: '{seed}' -> {generated}")
    print("-" * 30)
    
    # Early Stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_trump_model.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break


  0%|          | 0/20 [00:00<?, ?it/s]


Epoch 01 | Train Loss: 4.7723 | Val Loss: 4.1821
------------------------------
Generation samples:
Seed: 'I will' -> i will . … … … " via . com / <UNK> " thank you ! "
Seed: 'The fake news' -> the fake news ! … … . via … … via … " great ! " great .
Seed: 'Make America' -> make america . … via . … … pic . twitter . com / <UNK> " thanks
------------------------------

Epoch 02 | Train Loss: 3.9319 | Val Loss: 3.9232
------------------------------
Generation samples:
Seed: 'I will' -> i will ! ” djt / sr / trumped / <UNK> / 01 / <UNK> / <UNK>
Seed: 'The fake news' -> the fake news " " thanks ! " thanks . . . . " … ! ! "
Seed: 'Make America' -> make america bill . … via , … " true . " true . wish ! "
------------------------------

Epoch 03 | Train Loss: 3.5648 | Val Loss: 3.8328
------------------------------
Generation samples:
Seed: 'I will' -> i will deal ! … " ! djt ! " … … . " … . .
Seed: 'The fake news' -> the fake news ! " … " … " … ! " true ! ! ! ! "
Seed: 'Make America' -> make

In [ ]:
seed = "Democrats are"

print("Low Temp (0.2):", generate(model, seed, word2idx, idx2word, temperature=0.2))
print("Mid Temp (0.8):", generate(model, seed, word2idx, idx2word, temperature=0.8))
print("High Temp (1.5):", generate(model, seed, word2idx, idx2word, temperature=1.5))


Low Temp (0.2): democrats are ! ” " - president trump " - 2007 <UNK> . html " … " thanks . . . . " thanks . . . " thanks . . . . " thanks . . . . . . .
Mid Temp (0.8): democrats are ! ” " thanks . enjoy ! ! ! ! " thanks . <UNK> . will be back in the white house ! president djt " - president donald j . trump and president donald j . trump rumble %
High Temp (1.5): democrats are working ! … second report . … " by jeff ! log / article # museums america needs " running beaches . follow a question until now john needs debate at 1 . ‘ petrified it … congrats fighting an
